# 04_review_sentiment
This notebook runs sentiment analysis on customer reviews.


In [1]:
import pandas as pd

reviews = pd.read_csv(
    "../data/processed/reviews.csv"
)

print("Reviews:", len(reviews))
print("Columns:")
print(reviews.columns.tolist())

print(
    reviews.head()
)

Reviews: 5
Columns:
['review_id', 'product_id', 'user_id', 'rating', 'review_text']
  review_id product_id user_id  rating  \
0      r001       p001    u101       5   
1      r002       p001    u102       4   
2      r003       p002    u103       5   
3      r004       p003    u104       4   
4      r005       p004    u105       5   

                                         review_text  
0  Very comfortable and breathable. Great for wor...  
1  Looks stylish and fits well. Cushioning is goo...  
2  Excellent grip and durable quality for hiking ...  
3            Soft material and easy to wear all day.  
4  Great support during training sessions and ver...  


In [2]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import pipeline
import torch

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

print("Sentiment model loaded")

d:\DL_CP\multimodal-ecommerce-recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\DL_CP\multimodal-ecommerce-recommender\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aksha\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pyth

Sentiment model loaded


In [4]:
reviews["review_text"] = (
    reviews["review_text"]
    .fillna("")
    .astype(str)
)

In [6]:
from sentence_transformers import SentenceTransformer

In [7]:
review_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

review_embeddings = review_model.encode(
    reviews["review_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Review embedding shape:",
    review_embeddings.shape
)

Batches: 100%|██████████| 1/1 [00:00<00:00, 17.81it/s]

Review embedding shape: (5, 384)


In [9]:
import numpy as np

In [10]:
np.save(
    "../data/embeddings/review_embeddings.npy",
    review_embeddings
)

In [11]:
sentiments = []

texts = reviews["review_text"].tolist()

for i in range(0, len(texts), 32):

    batch = texts[i:i+32]

    result = sentiment_model(
        batch,
        truncation=True
    )

    sentiments.extend(result)

print("Sentiments:", len(sentiments))

Sentiments: 5


In [12]:
reviews["sentiment"] = [
    x["label"]
    for x in sentiments
]

reviews["sentiment_score"] = [
    x["score"]
    for x in sentiments
]

In [13]:
reviews.to_csv(
    "../data/processed/reviews_with_sentiment.csv",
    index=False
)